In [1]:
options(timeout = 600)
if (!require("remotes")) install.packages("remotes", repos="https://cloud.r-project.org")
remotes::install_github("bhklab/genefu", upgrade = "never")
library(genefu)

Loading required package: remotes


Installing 10 packages: S7, iC10TrainingData, impute, BiocFileCache, rmeta, survivalROC, AIMS, iC10, biomaRt, survcomp

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpXWrf7Y/remotes59204f9060f3/bhklab-genefu-9c9b66d/DESCRIPTION’ ... OK
* preparing ‘genefu’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* looking to see if a ‘data/datalist’ file should be added
* building ‘genefu_2.39.1.tar.gz’



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: survcomp

Loading required package: survival

Loading required package: prodlim

Loading required package: biomaRt

Loading required package: iC10

Loading required package: AIMS

Loading required package: e1071

Loading required package: Biobase

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min


Welcome to Bioconduc

In [3]:
BiocManager::install(c("GEOquery", "Biobase", "AnnotationDbi", "hgu133plus2.db"),
                      update = FALSE, ask = FALSE)
library(GEOquery)
library(Biobase)
library(AnnotationDbi)
library(hgu133plus2.db)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: http://cran.rstudio.com/

Bioconductor version 3.20 (BiocManager 1.30.27), R 4.4.0 (2024-04-24)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'GEOquery' 'Biobase' 'AnnotationDbi'
  'hgu133plus2.db'”
Setting options('download.file.method.GEOquery'='auto')

Setting options('GEOquery.inmemory.gpl'=FALSE)

Loading required package: stats4

Loading required package: IRanges

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: org.Hs.eg.db







In [ ]:
## ============================================================
## — Download GSE21653 and extract expression + phenotype data
## ============================================================
gset <- getGEO("GSE21653", GSEMatrix = TRUE, getGPL = FALSE)
eset <- gset[[1]]  # GSE21653 is single-platform (GPL570)

expr_mat <- exprs(eset)          # probes (rows) x samples (columns)
pheno    <- pData(eset)          # sample metadata, incl. original 'molecular subtype'

cat("Expression matrix dimensions (probes x samples):", dim(expr_mat), "\n")
cat("Value range:", range(expr_mat, na.rm = TRUE), "\n")


Found 1 file(s)

GSE21653_series_matrix.txt.gz



Expression matrix dimensions (probes x samples): 54613 266 
Value range: 1.6 14.42 


In [ ]:
## ============================================================
## — Log2 transform if not already log-scale
## GEO series matrices are sometimes linear-scale; PAM50 centroid
## correlation assumes log2 data.
## ============================================================
if (max(expr_mat, na.rm = TRUE) > 30) {
  cat("Values look linear-scale — applying log2(x + 1) transform.\n")
  expr_mat <- log2(expr_mat + 1)
} else {
  cat("Values already look log2-scale — no transform applied.\n")
}

Values already look log2-scale — no transform applied.


In [ ]:
## ============================================================
## — Map Affymetrix probe IDs -> Entrez Gene IDs (GPL570 / hgu133plus2)
## ============================================================
probe_ids  <- rownames(expr_mat)
entrez_ids <- mapIds(
  hgu133plus2.db,
  keys      = probe_ids,
  column    = "ENTREZID",
  keytype   = "PROBEID",
  multiVals = "first"
)

annot_df <- data.frame(
  probe          = probe_ids,
  EntrezGene.ID  = entrez_ids,
  row.names      = probe_ids,
  stringsAsFactors = FALSE
)

n_mapped <- sum(!is.na(annot_df$EntrezGene.ID))
cat(sprintf("Mapped %d / %d probes to an Entrez Gene ID.\n", n_mapped, length(probe_ids)))

'select()' returned 1:many mapping between keys and columns



Mapped 44640 / 54613 probes to an Entrez Gene ID.


In [ ]:
## ============================================================
## — Reformat for genefu (samples in ROWS, probes in COLUMNS)
## ============================================================
data_for_genefu <- t(expr_mat)   # transpose: samples x probes
cat("genefu input dimensions (samples x probes):", dim(data_for_genefu), "\n")

genefu input dimensions (samples x probes): 266 54613 


In [10]:
data(package = "genefu")$results[, "Item"]

[1] "annot.expos (expos)" "annot.nkis (nkis)"   "annot.vdxs (vdxs)"  
 [4] "claudinLowData"      "data.expos (expos)"  "data.nkis (nkis)"   
 [7] "data.vdxs (vdxs)"    "demo.expos (expos)"  "demo.nkis (nkis)"   
[10] "demo.vdxs (vdxs)"    "mod1"                "mod2"               
[13] "modelOvcAngiogenic"  "pam50"               "pam50.robust"       
[16] "pam50.scale"         "scmgene.robust"      "scmod1.robust"      
[19] "scmod2.robust"       "sig.endoPredict"     "sig.gene70"         
[22] "sig.gene76"          "sig.genius"          "sig.ggi"            
[25] "sig.oncotypedx"      "sig.pik3cags"        "sig.tamr13"         
[28] "sigOvcAngiogenic"    "sigOvcCrijns"        "sigOvcSpentzos"     
[31] "sigOvcTCGA"          "sigOvcYoshihara"     "ssp2003"            
[34] "ssp2003.robust"      "ssp2003.scale"       "ssp2006"            
[37] "ssp2006.robust"      "ssp2006.scale"

In [ ]:
## ============================================================
## — Run PAM50 intrinsic subtyping
## ============================================================
data(pam50)
data(pam50.robust)

pam50_result <- molecular.subtyping(
  sbt.model  = "pam50",
  data       = data_for_genefu,
  annot      = annot_df,
  do.mapping = TRUE,
  verbose    = TRUE
)

# Inspect the result structure the first time you run this —
# field names can vary slightly by genefu version.
str(pam50_result, max.level = 1)

genefu_calls <- as.character(pam50_result$subtype)
names(genefu_calls) <- rownames(data_for_genefu)

cat("\nRaw genefu PAM50 subtype call counts:\n")
print(table(genefu_calls, useNA = "ifany"))

List of 3
 $ subtype      : Factor w/ 5 levels "Basal","Her2",..: 3 2 3 2 3 4 3 4 3 1 ...
  ..- attr(*, "names")= chr [1:266] "GSM540108" "GSM540109" "GSM540110" "GSM540111" ...
 $ subtype.proba: num [1:266, 1:5] 0 0.1902 0.0371 0 0 ...
  ..- attr(*, "dimnames")=List of 2
 $ subtype.crisp: num [1:266, 1:5] 0 0 0 0 0 0 0 0 0 1 ...
  ..- attr(*, "dimnames")=List of 2

Raw genefu PAM50 subtype call counts:
genefu_calls
 Basal   Her2   LumA   LumB Normal 
    72     31     72     83      8 


In [ ]:
## ============================================================
## — Map genefu's naming to this project's class naming
## genefu's pam50 typically returns: Basal, Her2, LumA, LumB, Normal
## Your classifier has no "Normal" class — those samples cannot be
## compared to a model prediction and should be EXCLUDED from this
## sensitivity analysis, not forced into one of the four classes.
## ============================================================
subtype_map <- c(
  "Basal" = "Basal",
  "Her2"  = "Her2",
  "LumA"  = "Luminal A",
  "LumB"  = "Luminal B",
  "Normal" = NA  # excluded — no corresponding class in the trained model
)

genefu_mapped <- subtype_map[genefu_calls]
n_normal <- sum(genefu_calls == "Normal", na.rm = TRUE)
cat(sprintf(
  "\n%d of %d samples called 'Normal-like' by PAM50 — excluded from comparison "
  , n_normal, length(genefu_calls)
))
cat("(no corresponding class in the trained 4-class model).\n")


8 of 266 samples called 'Normal-like' by PAM50 — excluded from comparison (no corresponding class in the trained 4-class model).


In [ ]:
## ============================================================
## — Save results: sample ID, original IHC-surrogate label,
## new genefu intrinsic label, and whether they agree
## ============================================================
comparison_df <- data.frame(
  sample_id             = rownames(data_for_genefu),
  ihc_surrogate_subtype = pheno[rownames(data_for_genefu), "molecular subtype:ch1"],
  genefu_pam50_subtype  = genefu_mapped,
  stringsAsFactors      = FALSE
)

comparison_df$labels_agree <- comparison_df$ihc_surrogate_subtype == comparison_df$genefu_pam50_subtype

write.csv(
  comparison_df,
  "results/external_validation/gse21653_genefu_intrinsic_labels.csv",
  row.names = FALSE
)
cat("\nSaved: results/external_validation/gse21653_genefu_intrinsic_labels.csv\n")



Saved: /kaggle/working/gse21653_genefu_intrinsic_labels.csv


In [ ]:
ihc_canonical_map <- c(
  "Basal"    = "Basal",
  "ERBB2"    = "Her2",
  "LuminalA" = "Luminal A",
  "LuminalB" = "Luminal B"
)

comparison_df$ihc_surrogate_subtype_canonical <- ihc_canonical_map[comparison_df$ihc_surrogate_subtype]
comparison_df$labels_agree <- comparison_df$ihc_surrogate_subtype_canonical == comparison_df$genefu_pam50_subtype

write.csv(
  comparison_df,
  "results/external_validation/gse21653_genefu_intrinsic_labels.csv",
  row.names = FALSE
)

usable <- comparison_df[!is.na(comparison_df$genefu_pam50_subtype), ]
concordance_pct <- 100 * mean(usable$labels_agree, na.rm = TRUE)
cat(sprintf("Corrected concordance = %.1f%% (n=%d)\n", concordance_pct, nrow(usable)))

Corrected concordance = 75.4% (n=258)


In [ ]:
## ============================================================
## — Summary: IHC-surrogate vs genefu-intrinsic concordance
## (context for how much of the original 65% gap this label mismatch
## alone might explain)
## ============================================================
usable <- comparison_df[!is.na(comparison_df$genefu_pam50_subtype), ]
concordance_pct <- 100 * mean(usable$labels_agree, na.rm = TRUE)

cat(sprintf(
  "\nGSE21653: IHC-surrogate vs genefu-intrinsic label concordance = %.1f%% (n=%d)\n",
  concordance_pct, nrow(usable)
))
cat("Compare this to the ~60-62% concordance reported in independent IHC-vs-PAM50\n")
cat("literature -- if similar, it confirms the two label systems disagree here\n")
cat("for the same reason they disagree generally, not because of anything\n")
cat("specific to this dataset or your model.\n")

cat("\nNext step: bring 'gse21653_genefu_intrinsic_labels.csv' back into your\n")
cat("Python notebook, join by sample_id (GSM accession) with your EXISTING\n")
cat("model predictions, and recompute accuracy/F1/confusion matrix against\n")
cat("the 'genefu_pam50_subtype' column -- no need to re-run the frozen model,\n")
cat("only the ground-truth labels are changing.\n")



GSE21653: IHC-surrogate vs genefu-intrinsic label concordance = 75.4% (n=258)
Compare this to the ~60-62% concordance reported in independent IHC-vs-PAM50
literature -- if similar, it confirms the two label systems disagree here
for the same reason they disagree generally, not because of anything
specific to this dataset or your model.

Next step: bring 'gse21653_genefu_intrinsic_labels.csv' back into your
Python notebook, join by sample_id (GSM accession) with your EXISTING
model predictions, and recompute accuracy/F1/confusion matrix against
the 'genefu_pam50_subtype' column -- no need to re-run the frozen model,
only the ground-truth labels are changing.
